# Session 10: k-Means Clustering from Scratch

## Overview

Part 3 is about **clustering**: finding groups in data that has **no labels**. Nobody tells the algorithm what the
groups are, or even how many there are; it has to discover structure on its own. That is what makes clustering
useful (customer segments, document groups, anomaly hunting) and also what makes it hard to evaluate.

To be able to check our work, we start with data where the answer is secretly known: 178 Italian wines from three
grape varieties (*cultivars*), described by 13 chemical measurements. We hide the variety, cluster the wines, and
only at the end see whether the clusters recovered the varieties. You will build **k-means** by hand, see how it
can get stuck, why scaling decides the result, and how to choose the number of clusters.

By the end, you will be able to:

- Explain and implement Lloyd's algorithm for k-means: assign, update, repeat.
- Explain the inertia (within-cluster sum of squares) that k-means minimises.
- Recognise local minima, and use k-means++ initialisation and several restarts to avoid them.
- Show why features must be scaled before clustering.
- Choose k with the elbow method and the silhouette score.
- Compare clusters with known labels using the adjusted Rand index, and describe clusters with their averages.

**Prerequisites:** Session 1 (standardising), basic NumPy.

**Dataset:** [UCI Wine](https://archive.ics.uci.edu/dataset/109/wine). Chemical analysis of 178 wines grown in
the same region of Italy from three different cultivars: alcohol, malic acid, ash, alkalinity of ash, magnesium,
total phenols, flavanoids, non-flavanoid phenols, proanthocyanins, colour intensity, hue, OD280/OD315 (a protein
measure) and proline (an amino acid). It is downloaded automatically by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering **← you are here** |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** Without being told, can k-means find the three grape varieties in the chemistry
of the wines?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| numpy | k-means by hand |
| pandas | Tables |
| scikit-learn | Scaling, the reference k-means, PCA for plotting, metrics |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from importlib.metadata import version

for package in ["ucimlrepo", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the wines and hide the answer

We keep the cultivar in a separate variable, `cultivar`, and do **not** give it to any algorithm. It is used only
in Step 9, to check the result. In a real clustering problem, it would not exist.

In [ ]:
from ucimlrepo import fetch_ucirepo

wine = fetch_ucirepo(id=109).data.original
cultivar = wine["class"]                         # hidden until Step 9
X = wine.drop(columns="class")
print(f"{len(X)} wines, {X.shape[1]} measurements")
print(X.describe().round(2).T[["mean", "std", "min", "max"]])

**Read the output:** the measurements are on wildly different scales. **Proline** has a standard deviation of
about 315, **magnesium** about 14, while **hue** and **non-flavanoid phenols** vary by about 0.1-0.2. Keep this in
mind: k-means is based on distances, and Step 8 shows what these scales do to it.

## Step 3: Standardise, and get a picture

As in Session 1, standardise each measurement to mean 0 and standard deviation 1.

To *see* 13-dimensional data, we project it onto 2 dimensions with **PCA** (principal component analysis), which
finds the two directions along which the wines differ most. PCA is used here only for plotting; the clustering
always uses all 13 standardised measurements.

In [ ]:
Z = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(Z)
picture = pca.transform(Z)
print(f"the 2-D picture keeps {pca.explained_variance_ratio_.sum():.0%} of the variation")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*picture.T, s=15, c="grey")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("178 wines, no labels")
plt.show()

**Read the output:** the picture keeps 55% of the variation, enough to see shape. Even without colours, the wines
are not one uniform blob: there are denser regions to the left, the right and the bottom. Whether they are three
groups is what k-means will try to decide.

## Step 4: The k-means algorithm

k-means looks for k **centroids** (cluster centres) such that every point is close to its own centroid. It
minimises the **inertia**, the total squared distance from each point to its centroid:

$$\text{inertia} = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2$$

**Lloyd's algorithm** finds a solution by repeating two easy steps:

1. **Assign:** put every point in the cluster of its nearest centroid.
2. **Update:** move every centroid to the mean of the points assigned to it.

Each step can only lower the inertia, so the algorithm always stops, usually after a handful of rounds. It starts
from k randomly chosen points as the first centroids.

In [ ]:
def kmeans(Z, k, seed, max_rounds=100):
    rng = np.random.default_rng(seed)
    centroids = Z[rng.choice(len(Z), size=k, replace=False)]          # k random points to start
    history = []
    for round_number in range(1, max_rounds + 1):
        distances = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)   # n x k squared distances
        labels = distances.argmin(axis=1)                                         # 1. assign
        history.append(distances.min(axis=1).sum())                               #    inertia
        new_centroids = np.array([Z[labels == j].mean(axis=0) for j in range(k)])  # 2. update
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids
    return labels, centroids, history


labels, centroids, history = kmeans(Z, k=3, seed=0)
print(f"stopped after {len(history)} rounds")
print("inertia by round:", np.round(history, 1))
print("cluster sizes:", np.bincount(labels))

**Read the output:** the inertia falls in every round, from 3,131 to 1,278, and the algorithm stops after 7
rounds, when no point changes cluster. The three clusters have similar sizes (62, 65 and 51 wines).

`Z[:, None, :] - centroids[None, :, :]` uses NumPy **broadcasting** to compute the difference between every point
and every centroid at once, giving a 178 × 3 × 13 array, without a Python loop.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*picture.T, s=15, c=labels, cmap="viridis")
ax.scatter(*pca.transform(centroids).T, s=250, marker="X", c="red", edgecolor="white", label="centroids")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("k-means clusters, k = 3")
ax.legend()
plt.show()

**Read the output:** the three clusters occupy the three dense regions of the picture, with the centroids in the
middle of each.

## Step 5: Local minima: a bad start gives a bad answer

Lloyd's algorithm only ever goes downhill from where it starts, so it can get stuck in a **local minimum**: a
clustering that no single round can improve, but that is not the best. Run it from ten different random starts.

In [ ]:
for seed in range(10):
    labels_s, _, history_s = kmeans(Z, k=3, seed=seed)
    print(f"seed {seed}: final inertia {history_s[-1]:7.1f}   sizes {np.bincount(labels_s)}")

**Read the output:** most starts end with an inertia between 1,277.9 and 1,282.5, with clusters of about 50-67
wines. **Seed 6** gets stuck at 1,583.4, with clusters of 74, 92 and just **12** wines. Its starting centroids fell
badly, and once a centroid has settled among a few odd wines, no single assign-update round can move it far
enough to find a better place. Even the "good" runs differ slightly from one another.

Two standard defences, both built into scikit-learn:

- **k-means++ initialisation:** choose the first centroid at random, then each next one with a probability that
  grows with its distance from the centroids already chosen. Starting centroids are spread out.
- **Several restarts (`n_init`):** run the algorithm several times and keep the result with the lowest inertia.

## Step 6: The same in scikit-learn

In [ ]:
sk = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=0).fit(Z)
print(f"scikit-learn inertia: {sk.inertia_:.1f}  (our best: {min(kmeans(Z, 3, s)[2][-1] for s in range(10)):.1f})")
print("cluster sizes:", np.bincount(sk.labels_))

**Read the output:** scikit-learn reaches the same lowest inertia as our best run (1,277.9). From here on we use
`KMeans`, which is much faster on large data.

## Step 7: How many clusters?

We told k-means to find 3 clusters. In a real problem, nobody knows k. Two common guides:

- **The elbow method:** inertia always falls as k grows (more centroids, shorter distances; with k = n it is 0).
  Look for the k after which it falls much more slowly: the "elbow".
- **The silhouette score:** for each point, compare a = its average distance to its own cluster with b = its
  average distance to the nearest *other* cluster: s = (b - a) / max(a, b). Near 1: well inside its cluster; near
  0: on a border; negative: probably in the wrong cluster. Average over all points; higher is better.

In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(Z, model.labels_)})
choice = pd.DataFrame(rows).set_index("k")
print(choice.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
choice["inertia"].plot(ax=axes[0], marker="o", title="Elbow method")
choice["silhouette"].plot(ax=axes[1], marker="o", title="Silhouette score (higher is better)")
plt.show()

**Read the output:** the inertia drops steeply from k = 2 to k = 3 (1,659 to 1,278), then much more slowly: the
elbow is at **3**. The silhouette score is also highest at k = 3 (0.285). Both point to three clusters.

On real data the two guides often disagree, or show no clear elbow at all. They are guides, not rules: the
final choice also depends on what the clusters are for (Session 11).

A silhouette score of 0.285 is modest: the clusters are real but not far apart, as the picture in Step 4 shows.

## Step 8: What scaling does

Repeat the clustering on the **raw, unscaled** measurements.

In [ ]:
raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
centres = pd.DataFrame(raw.cluster_centers_, columns=X.columns)
print("raw-data centroids: spread of each measurement across the three centroids")
print((centres.max() - centres.min()).sort_values(ascending=False).round(2).head(5).to_string())

**Read the output:** on raw data, the three centroids differ by 737 in proline, by 13 in magnesium, and by less
than 4 in everything else. Proline's numbers are so much larger that it dominates every distance: k-means on raw data is
essentially **clustering on proline alone**, ignoring the other 12 measurements. Step 9 shows the cost.

**Always scale before clustering** (unless all features share a unit and their spread is meaningful).

## Step 9: Did k-means find the cultivars?

Now unhide the cultivar. A cross-tabulation shows how the clusters line up with the varieties. The cluster
numbers are arbitrary (cluster 0 is not "cultivar 1"), so we look for a one-to-one match.

The **adjusted Rand index (ARI)** summarises the agreement between two groupings, ignoring the numbering: 1 means
identical, 0 means no better than random.

In [ ]:
print("scaled data:")
print(pd.crosstab(cultivar, sk.labels_, rownames=["cultivar"], colnames=["cluster"]))
print(f"ARI = {adjusted_rand_score(cultivar, sk.labels_):.3f}")

print("\nraw data:")
print(pd.crosstab(cultivar, raw.labels_, rownames=["cultivar"], colnames=["cluster"]))
print(f"ARI = {adjusted_rand_score(cultivar, raw.labels_):.3f}")

**Read the output:** on scaled data, each cluster is almost exactly one cultivar: all 59 wines of cultivar 1 in
one cluster, all 48 of cultivar 3 in another, and 65 of the 71 wines of cultivar 2 in the third. Only **6 of 178**
wines are grouped with another variety. ARI = 0.897. Without ever seeing a label, k-means rediscovered the grape
varieties from the chemistry.

On raw data, every cluster mixes cultivars, and ARI falls to 0.371. Same algorithm, same data; only the scaling
differs.

Remember that this check is a luxury. With real unlabelled data you cannot compute ARI, which is why the elbow,
the silhouette and, above all, **whether the clusters make sense** to someone who knows the domain are what you
have.

## Step 10: Describe the clusters

A clustering is only useful if you can say what each cluster *is*. Average each measurement per cluster, in the
original units.

In [ ]:
pd.set_option("display.width", 120)
profile = X.groupby(sk.labels_).mean().T
profile.columns = [f"cluster {c} (n={n})" for c, n in zip(profile.columns, np.bincount(sk.labels_))]
z_profile = pd.DataFrame(Z, columns=X.columns).groupby(sk.labels_).mean().T
distinctive = z_profile.abs().max(axis=1).sort_values(ascending=False).index[:6]
print(profile.loc[distinctive].round(2))

**Read the output:** the six measurements that differ most between clusters (in standard deviations) describe
them. Cluster 2 (cultivar 1) has **high proline** (about 1,100), flavanoids, phenols and OD280. Cluster 1
(cultivar 3) has **low flavanoids** (0.82), OD280 and hue, but the **deepest colour** (7.2). Cluster 0 (cultivar 2)
has the **palest colour** (3.0) and the **least proline** (510). A wine chemist could now name the clusters, which is the real test of a clustering.

## What this session hands to the next

- **k-means from first principles:** inertia, assign-update rounds, local minima, k-means++ and restarts.
- **Practical rules:** scale first; choose k with the elbow and silhouette, then sanity-check with meaning.
- **Validation:** ARI when labels exist; cluster profiles always.
- Session 11 applies clustering to a real business problem with no hidden answer: segmenting a wholesaler's
  customers, and compares k-means with hierarchical clustering.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Clustering unscaled data | The largest-valued feature decides everything (ARI 0.37) | `StandardScaler` first |
| A single random start | Stuck in a local minimum (seed 6) | `init="k-means++"`, `n_init=10` or more |
| Choosing k by lowest inertia | Inertia always falls; k = n gives 0 | Elbow, silhouette, and domain sense |
| Reading cluster numbers as meaningful | Cluster 0 in one run can be cluster 2 in the next | Match clusters by profile or cross-tab; use ARI |
| Clustering on the 2-D PCA picture | Throws away 45% of the variation | Plot with PCA, cluster on all features |
| Expecting k-means to find any shape | It assumes round, similar-sized clusters | Try other algorithms for long, curved or uneven groups (Session 11) |

## Exercises

- [ ] Write the k-means++ initialisation yourself and use it in `kmeans`. Does seed 6 still get stuck?
- [ ] Cluster with only the two measurements `Flavanoids` and `Proline` (scaled). Plot them directly and compute
  the ARI. How much do the other 11 measurements add?
- [ ] Run `KMeans(n_clusters=2)` and `KMeans(n_clusters=4)` and cross-tabulate with the cultivar. Which cultivars
  merge at k = 2? Which splits at k = 4?
- [ ] Add a 14th column of pure random noise (scaled) and re-cluster. What happens to the ARI? Add five noise
  columns.

Next, Session 11 is the Part 3 bootcamp: segmenting a wholesale distributor's customers.